# CattleSSFR hardening_v2 on Colab Pro+

This notebook executes the complete 29-training hardening matrix and all 16 Holstein2025 controls sequentially. Completed Drive artifacts are validated and skipped automatically; no shard index is required.


## Run configuration


In [ ]:
RUN_SMOKE = False
RUN_HARDENING_V2 = True
RUN_POSTPROCESS_AUDITS = True
BUILD_FINAL_EVIDENCE = True

HARDENING_CONFIG = 'configs/cattlessfr_hardening_v2_colab_proplus.yaml'
HARDENING_MATRIX_CONFIG = 'configs/experiment_matrix_hardening_v2.yaml'
HOLSTEIN_MATRIX_CONFIG = 'configs/experiment_matrix_holstein_hardening_v2.yaml'
HARDENING_SCOPE = 'configs/final_evidence_scope_hardening_v2.yaml'
HOLSTEIN_METADATA = 'artifacts/metadata/holstein2025_open_set.csv'

print({
    'RUN_HARDENING_V2': RUN_HARDENING_V2,
    'RUN_POSTPROCESS_AUDITS': RUN_POSTPROCESS_AUDITS,
    'BUILD_FINAL_EVIDENCE': BUILD_FINAL_EVIDENCE,
    'HARDENING_MATRIX_CONFIG': HARDENING_MATRIX_CONFIG,
    'HOLSTEIN_MATRIX_CONFIG': HOLSTEIN_MATRIX_CONFIG,
    'HARDENING_SCOPE': HARDENING_SCOPE,
})


## Google Drive setup


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import os
import zipfile

drive.mount('/content/drive')

BUNDLE_PATH = Path('/content/drive/MyDrive/cattle_face_identification_hardening_v2_bundle.zip')
BUNDLE_SHA_PATH = Path('/content/drive/MyDrive/cattle_face_identification_hardening_v2_bundle.sha256.txt')
BUNDLE_MARKER = Path('/content/drive/MyDrive/cattle_face_identification/.colab_hardening_v2_bundle.sha256')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if BUNDLE_PATH.exists():
    bundle_sha = sha256_file(BUNDLE_PATH)
    if BUNDLE_SHA_PATH.exists():
        expected_sha = BUNDLE_SHA_PATH.read_text(encoding='utf-8').split()[0]
        assert bundle_sha == expected_sha, 'Colab hardening bundle SHA-256 mismatch'
    applied_sha = BUNDLE_MARKER.read_text(encoding='utf-8').strip() if BUNDLE_MARKER.exists() else ''
    if applied_sha != bundle_sha:
        with zipfile.ZipFile(BUNDLE_PATH) as archive:
            members = archive.namelist()
            assert members and all(name.startswith('cattle_face_identification/') and '..' not in Path(name).parts for name in members)
            archive.extractall('/content/drive/MyDrive')
        BUNDLE_MARKER.parent.mkdir(parents=True, exist_ok=True)
        BUNDLE_MARKER.write_text(bundle_sha + '\n', encoding='utf-8')
        print('Applied verified hardening bundle:', bundle_sha)
    else:
        print('Hardening bundle already applied:', bundle_sha)
else:
    print('No deployment bundle found; using the existing Drive project tree.')

PROJECT_DIR = Path('/content/drive/MyDrive/cattle_face_identification')
assert (PROJECT_DIR / 'pyproject.toml').is_file(), f'Project not found: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
PERSISTENT_OUTPUT_DIRS = [
    Path('artifacts/runs'), Path('artifacts/metadata'), Path('artifacts/matrix'),
    Path('artifacts/audits'), Path('artifacts/evidence'), Path('artifacts/notebook_logs'),
    Path('thesis/tables/hardening_v2'), Path('thesis/figures/hardening_v2'),
]
project_root = PROJECT_DIR.resolve()
assert str(project_root).startswith('/content/drive/'), 'Persistent project is not on Google Drive'
for relative_dir in PERSISTENT_OUTPUT_DIRS:
    output_dir = (PROJECT_DIR / relative_dir).resolve()
    try:
        output_dir.relative_to(project_root)
    except ValueError as exc:
        raise AssertionError(f'Persistent output escaped Google Drive project: {output_dir}') from exc
    output_dir.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT_DIR)
print('Persistent outputs:', [str(path) for path in PERSISTENT_OUTPUT_DIRS])


## Helpers and environment


In [ ]:
import os
import queue
import shlex
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path

COMMAND_HEARTBEAT_SECONDS = 15
LOG_DIR = Path('artifacts/notebook_logs')
LATEST_LOG = LOG_DIR / 'latest_command.log'

def _pipe_reader(pipe, output_queue):
    try:
        for line in iter(pipe.readline, ''):
            output_queue.put(line)
    finally:
        output_queue.put(None)

def run_cmd(args, check=True):
    args = [str(arg) for arg in args]
    command_text = ' '.join(shlex.quote(arg) for arg in args)
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    safe_name = ''.join(ch if ch.isalnum() else '_' for ch in command_text).strip('_')[:120]
    log_path = LOG_DIR / f"{time.strftime('%Y%m%d_%H%M%S')}_{safe_name or 'command'}.log"
    print('+', command_text, flush=True)
    env = dict(os.environ)
    env['PYTHONUNBUFFERED'] = '1'
    start = time.time()
    with log_path.open('w', encoding='utf-8', buffering=1) as command_log, LATEST_LOG.open('w', encoding='utf-8', buffering=1) as latest_log:
        process = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        output_queue = queue.Queue()
        reader = threading.Thread(target=_pipe_reader, args=(process.stdout, output_queue), daemon=True)
        reader.start()
        while True:
            try:
                item = output_queue.get(timeout=COMMAND_HEARTBEAT_SECONDS)
            except queue.Empty:
                item = f'[notebook] still running after {int(time.time() - start)}s: {command_text}\n'
            if item is None:
                break
            print(item, end='', flush=True)
            command_log.write(item)
            latest_log.write(item)
        return_code = process.wait()
        reader.join(timeout=1)
        footer = f'[notebook] exit_code={return_code} elapsed_seconds={int(time.time() - start)}\n'
        print(footer, end='', flush=True)
        command_log.write(footer)
        latest_log.write(footer)
    if check and return_code != 0:
        raise subprocess.CalledProcessError(return_code, args)

run_cmd([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'])
run_cmd([sys.executable, '-m', 'pip', 'install', '-e', '.'])
source_root = (PROJECT_DIR / 'src').resolve()
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))
run_cmd([sys.executable, '-u', '-c', "import cattle_id, pathlib; print(pathlib.Path(cattle_id.__file__).resolve())"])
gpu_inventory = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    check=False, capture_output=True, text=True,
).stdout.strip()
print('GPU inventory:', gpu_inventory)
if RUN_HARDENING_V2:
    assert any(name in gpu_inventory.upper() for name in ('A100', 'H100')), f'hardening_v2 requires A100/H100; detected: {gpu_inventory or "none"}'


## Prepare pinned Holstein2025 metadata


In [ ]:
import yaml
from cattle_id.colab_runtime import write_holstein_runtime_configs

HOLSTEIN_ROOT = Path('/content/cattle_runtime/raw/Holstein2025')
readiness_source = Path('configs/holstein2025_open_set.yaml')
training_source = Path('configs/holstein2025_colab_proplus.yaml')
expected_commit = str(yaml.safe_load(readiness_source.read_text(encoding='utf-8'))['dataset']['source_commit'])
run_cmd(['git', 'lfs', 'install', '--skip-repo'], check=False)
if not (HOLSTEIN_ROOT / 'datasets_v2').exists():
    if HOLSTEIN_ROOT.exists():
        shutil.rmtree(HOLSTEIN_ROOT)
    HOLSTEIN_ROOT.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(['git', 'clone', '--depth', '1', 'https://github.com/JZM-shuimu/Cattle-ID.git', str(HOLSTEIN_ROOT)])
run_cmd(['git', '-C', str(HOLSTEIN_ROOT), 'lfs', 'pull'])
actual_commit = subprocess.run(
    ['git', '-C', str(HOLSTEIN_ROOT), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True,
).stdout.strip()
if actual_commit != expected_commit:
    run_cmd(['git', '-C', str(HOLSTEIN_ROOT), 'fetch', '--depth', '1', 'origin', expected_commit])
    run_cmd(['git', '-C', str(HOLSTEIN_ROOT), 'checkout', '--detach', expected_commit])
    run_cmd(['git', '-C', str(HOLSTEIN_ROOT), 'lfs', 'pull'])
    actual_commit = subprocess.run(
        ['git', '-C', str(HOLSTEIN_ROOT), 'rev-parse', 'HEAD'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
assert actual_commit == expected_commit, f'Holstein2025 commit mismatch: {actual_commit} != {expected_commit}'
runtime_configs = write_holstein_runtime_configs(
    readiness_source=readiness_source,
    training_source=training_source,
    runtime_dir=Path('/content/cattle_runtime/configs'),
    dataset_root=HOLSTEIN_ROOT,
)
HOLSTEIN_READINESS_CONFIG = str(runtime_configs.readiness)
run_cmd([sys.executable, 'tools/holstein2025_readiness.py', '--config', HOLSTEIN_READINESS_CONFIG])
assert Path(HOLSTEIN_METADATA).is_file(), 'Holstein2025 metadata was not written to Drive'

run_cmd([sys.executable, '-u', '-m', 'cattle_id.run_matrix', '--matrix', HARDENING_MATRIX_CONFIG, '--out', 'artifacts/matrix/experiment_matrix_hardening_v2_jobs.json'])
run_cmd([sys.executable, '-u', '-m', 'cattle_id.run_matrix', '--matrix', HOLSTEIN_MATRIX_CONFIG, '--out', 'artifacts/matrix/experiment_matrix_holstein_hardening_v2_jobs.json'])
training_jobs = json.loads(Path('artifacts/matrix/experiment_matrix_hardening_v2_jobs.json').read_text(encoding='utf-8'))
holstein_jobs = json.loads(Path('artifacts/matrix/experiment_matrix_holstein_hardening_v2_jobs.json').read_text(encoding='utf-8'))
assert len(training_jobs) == 29 and len(holstein_jobs) == 16
print('Verified contracts: 29 trainings and 16 Holstein evaluations')


## Execute all 29 hardening_v2 trainings sequentially


In [ ]:
if RUN_HARDENING_V2:
    run_cmd([
        sys.executable, '-u', '-m', 'cattle_id.hardening_matrix',
        '--matrix', HARDENING_MATRIX_CONFIG,
        '--progress', 'artifacts/matrix/hardening_v2_progress.json',
    ])
else:
    print('Skipping hardening_v2 training matrix')


## Run shortcut, robustness, Grad-CAM and Holstein controls


In [ ]:
if RUN_POSTPROCESS_AUDITS:
    run_cmd([
        sys.executable, '-u', '-m', 'cattle_id.hardening_postprocess',
        '--training-matrix', HARDENING_MATRIX_CONFIG,
        '--holstein-metadata', HOLSTEIN_METADATA,
        '--progress', 'artifacts/matrix/holstein_hardening_v2_progress.json',
    ])
else:
    print('Skipping hardening post-processing audits')


## Generate tables, figures and persistent evidence bundle


In [ ]:
if BUILD_FINAL_EVIDENCE:
    run_cmd([sys.executable, '-u', 'tools/finalize_hardening_colab.py'])
    print('Persistent evidence:', (PROJECT_DIR / 'artifacts/evidence/hardening_v2_evidence.zip').resolve())
    print('Persistent tables:', (PROJECT_DIR / 'thesis/tables/hardening_v2').resolve())
    print('Persistent figures:', (PROJECT_DIR / 'thesis/figures/hardening_v2').resolve())
else:
    print('Evidence generation disabled')
